In [ ]:
import requests
from pathlib import Path
def inspect_types(value, path="root"):
    if isinstance(value, dict):
        for key, item in value.items():
            inspect_types(item, f"{path}.{key}")

    elif isinstance(value, list):
        if value:
            inspect_types(value[0], f"{path}[0]")

    else:
        print(f"{path}: {type(value).__name__}")

all_neos=[]
dates=[("2026-07-17","2026-07-18"),("2026-07-19","2026-07-20")]
for start_date,end_date in dates:
    url=f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key=GehboWogq2EOuimvY5oOLjyd5yX44db2kkuFhd1c"
    response=requests.get(url)
    if response.status_code==200:
        payload=response.json()
        for date,objects in payload["near_earth_objects"].items():
            #for obj in objects:
             all_neos.extend(objects)
            # print(all_neos)
    else:
        print("error")
        break
  

neo_ids=[str(obj["neo_reference_id"])for obj in all_neos]
neo_ids=list(dict.fromkeys(neo_ids))
Path("data/raw").mkdir(parents=True, exist_ok=True)

Path("data/raw/extracted_ids.txt").write_text( "\n".join(neo_ids),encoding="utf-8")

print(f"Extracted {len(neo_ids)} unique NEO ids.")

print(all_neos[0]["close_approach_data"])

records=[]
for obj in all_neos:
    approaches=obj.get("close_approach_data",[])
    num_close_approaches=len(approaches)
    if approaches:
        approach=approaches[0]
        try:
          distance_km=float(approach["miss_distance"]["kilometers"])
          distance_lunar=float(approach["miss_distance"]["lunar"])
          velocity=float(approach["relative_velocity"]["kilometers_per_hour"])
        except (ValueError ,TypeError) as e:
          distance_km=None
          distance_lunar=None
          velocity=None
    else:
        distance_km=None
        distance_lunar=None
        velocity=None
    try:
        diameter_max=float(obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"])
    except (ValueError, TypeError, KeyError) as e:
        diameter_max=None
    record={"neo_id":obj.get("neo_reference_id"),"num_close_approaches":num_close_approaches,"miss_distance_km":distance_km,"miss_distance_lunar":distance_lunar,"relative_velocity_kph":velocity,"estimated_diameter_max_km":diameter_max}
    records.append(record)
print(len(records))
print(records[0])
for i, obj in enumerate(all_neos[:5]):
    print(f"\n--- Record {i + 1} ---")
    inspect_types(obj)





Extracted 19 unique NEO ids.
[{'close_approach_date': '2026-07-17', 'close_approach_date_full': '2026-Jul-17 18:28', 'epoch_date_close_approach': 1784312880000, 'relative_velocity': {'kilometers_per_second': '9.9718328384', 'kilometers_per_hour': '35898.5982183234', 'miles_per_hour': '22305.9928762688'}, 'miss_distance': {'astronomical': '0.4326166289', 'lunar': '168.2878686421', 'kilometers': '64718526.210020443', 'miles': '40214227.4284023934'}, 'orbiting_body': 'Earth'}]
19
{'neo_id': '3623521', 'num_close_approaches': 1, 'miss_distance_km': 64718526.210020445, 'miss_distance_lunar': 168.2878686421, 'relative_velocity_kph': 35898.5982183234, 'estimated_diameter_max_km': 0.1367854737}

--- Record 1 ---
root.links.self: str
root.id: str
root.neo_reference_id: str
root.name: str
root.nasa_jpl_url: str
root.absolute_magnitude_h: float
root.estimated_diameter.kilometers.estimated_diameter_min: float
root.estimated_diameter.kilometers.estimated_diameter_max: float
root.estimated_diameter.

In [4]:
def calculate_statistics(values):
    valid_values = []

    for value in values:
        if value is not None:
            valid_values.append(float(value))

    if not valid_values:
        return None, None, None

    value_min = valid_values[0]
    value_max = valid_values[0]
    total = 0

    for value in valid_values:
        if value < value_min:
            value_min = value

        if value > value_max:
            value_max = value

        total += value

    mean = total / len(valid_values)

    return value_min, value_max, mean

In [5]:
diameters = []

for record in records:
    diameters.append(record["estimated_diameter_max_km"])

diameter_min, diameter_max, diameter_mean = calculate_statistics(diameters)

print(f"Max diameter: {diameter_max} km")
print(f"Min diameter: {diameter_min} km")
print(f"Mean diameter: {diameter_mean} km")

Max diameter: 0.8129053443 km
Min diameter: 0.0156329154 km
Mean diameter: 0.15474024931052632 km


In [6]:
distances = []

for record in records:
    distances.append(record["miss_distance_km"])

distance_min, distance_max, distance_mean = calculate_statistics(distances)

print(f"Max miss distance: {distance_max} km")
print(f"Min miss distance: {distance_min} km")
print(f"Mean miss distance: {distance_mean} km")

Max miss distance: 74716629.7076368 km
Min miss distance: 8918882.14343415 km
Mean miss distance: 48321651.89724956 km


In [7]:
velocities = []

for record in records:
    velocities.append(record["relative_velocity_kph"])

velocity_min, velocity_max, velocity_mean = calculate_statistics(velocities)

print(f"Max velocity: {velocity_max} kph")
print(f"Min velocity: {velocity_min} kph")
print(f"Mean velocity: {velocity_mean} kph")

Max velocity: 96579.3857033081 kph
Min velocity: 12594.0999118889 kph
Mean velocity: 36186.336797801494 kph


In [11]:
def missing_percentage(records, field):
    missing_count = 0

    for record in records:
        value = record.get(field)

        if value is None or value == []:
            missing_count += 1

    percentage = (missing_count / len(records)) * 100

    return missing_count, percentage

In [12]:
close_count, close_percentage = missing_percentage( all_neos, "close_approach_data")

print(f"Missing/empty close_approach_data: {close_count}")
print(f"Percentage: {close_percentage:.2f}%")

Missing/empty close_approach_data: 0
Percentage: 0.00%


In [13]:
magnitude_count, magnitude_percentage = missing_percentage(all_neos,"absolute_magnitude_h")

print(f"Missing absolute_magnitude_h: {magnitude_count}")
print(f"Percentage: {magnitude_percentage:.2f}%")

Missing absolute_magnitude_h: 0
Percentage: 0.00%


In [14]:
def check_boolean_field(records, field):
    missing_count = 0
    non_boolean_count = 0

    for record in records:
        if field not in record:
            missing_count += 1
        elif not isinstance(record[field], bool):
            non_boolean_count += 1

    return missing_count, non_boolean_count

In [15]:
missing_hazardous, non_boolean_hazardous = check_boolean_field(
    all_neos,
    "is_potentially_hazardous_asteroid"
)

print(f"Missing field: {missing_hazardous}")
print(f"Non-boolean values: {non_boolean_hazardous}")

Missing field: 0
Non-boolean values: 0
